# Dashboard requested model contract v1

目标：把 `src/model` 中的 requested QuantBench 模型清单健康状态接入 dashboard export、view model、HTML 页面和 schedule refresh 通知路径。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT_DIR = ROOT / 'research' / 'dashboard_requested_model_contract_v1'
MATRIX_CARD = ROOT / 'research' / 'quantbench_requested_model_contract_v1' / 'matrix_card'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

assert MATRIX_CARD.exists(), MATRIX_CARD
ROOT

PosixPath('/Users/rziiiii/dev/workspace')

In [2]:
from src.dashboard import build_dashboard_view_model, write_dashboard_html, write_model_matrix_dashboard_export

dashboard_export_path = EXPERIMENT_DIR / 'dashboard_export.json'
dashboard_html_path = EXPERIMENT_DIR / 'dashboard.html'

dashboard_payload = write_model_matrix_dashboard_export(
    MATRIX_CARD,
    output_path=dashboard_export_path,
    top_n=8,
)
view_model = build_dashboard_view_model(dashboard_payload)
html = write_dashboard_html(view_model, output_path=dashboard_html_path, title='SOXL Requested Model Contract')

contract = dashboard_payload['requested_model_contract']
assert contract['schema_version'] == 'model.requested_contract_health.v1'
assert contract['requested_model_count'] == 32
assert contract['registered_model_count'] == 32
assert contract['completed_requested_model_count'] == 32
assert contract['missing_from_registry'] == []
assert contract['missing_from_ledger'] == []
assert contract['incomplete_requested_models'] == []
assert contract['all_requested_models_completed'] is True
assert 'Requested 模型' in html

contract

{'schema_version': 'model.requested_contract_health.v1',
 'requested_model_count': 32,
 'registered_model_count': 32,
 'ledger_model_count': 32,
 'completed_requested_model_count': 32,
 'missing_from_registry': [],
 'extra_in_registry': [],
 'missing_from_ledger': [],
 'incomplete_requested_models': [],
 'all_requested_models_completed': True,
 'adapter_statuses': ['paper_strict_adapter']}

In [3]:
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

events: list[str] = []

def reuse_existing_matrix_card(**kwargs):
    return {
        'status': 'existing_matrix_card_reused',
        'variant_count': contract['ledger_model_count'],
    }

config = ModelDashboardPipelineConfig(
    name='dashboard_requested_model_contract_refresh',
    card_dir=MATRIX_CARD,
    dashboard_output_path=EXPERIMENT_DIR / 'scheduled_dashboard_export.json',
    dashboard_page_output_path=EXPERIMENT_DIR / 'scheduled_dashboard.html',
    dashboard_page_title='SOXL Requested Contract Scheduled View',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=(),
    dashboard_top_n=8,
    notify_details=True,
)

schedule_result = run_model_dashboard_refresh_once(
    config,
    lock_dir=EXPERIMENT_DIR / 'locks',
    notify=events.append,
    matrix_runner=reuse_existing_matrix_card,
)

assert schedule_result.status == 'success'
assert schedule_result.payload['requested_model_count'] == 32
assert schedule_result.payload['requested_model_completed_count'] == 32
assert schedule_result.payload['requested_model_missing_count'] == 0
assert schedule_result.payload['requested_model_incomplete_count'] == 0
assert schedule_result.payload['requested_model_all_completed'] is True
assert events and 'requested_models=32/32 missing=0 incomplete=0' in events[0]

schedule_result.payload

{'matrix_card': '/Users/rziiiii/dev/workspace/research/quantbench_requested_model_contract_v1/matrix_card',
 'dashboard_export': '/Users/rziiiii/dev/workspace/research/dashboard_requested_model_contract_v1/scheduled_dashboard_export.json',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_artifact_id': 'matrix_card',
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_manifest_path': '/Users/rziiiii/dev/workspace/research/quantbench_requested_model_contract_v1/matrix_card/analysis_manifest.json',
 'dashboard_manifest_hash': '09148e577c8da24ad3c79bbb2958662163ca81e64411377283f35c0955d224fc',
 'dashboard_manifest_producer': 'src.analysis.exports',
 'dashboard_page': '/Users/rziiiii/dev/workspace/research/dashboard_requested_model_contract_v1/scheduled_dashboard.html',
 'dashboard_page_schema_version': 'dashboard.static_page.v1',
 'dashboard_view_model_schema_version': 'dashboard.view_model.v1',
 'dashboard_page_model_card_count': 8,
 'top_model': 'mlp_mixer_m

In [4]:
summary = {
    'schema_version': 'research.dashboard_requested_model_contract.v1',
    'dashboard_export_path': str(dashboard_export_path.relative_to(ROOT)),
    'dashboard_html_path': str(dashboard_html_path.relative_to(ROOT)),
    'scheduled_dashboard_export_path': str(config.dashboard_output_path.relative_to(ROOT)),
    'scheduled_dashboard_html_path': str(config.dashboard_page_output_path.relative_to(ROOT)),
    'requested_model_contract': contract,
    'schedule_payload': schedule_result.payload,
    'notification': events[0],
}
(EXPERIMENT_DIR / 'analysis_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
summary

{'schema_version': 'research.dashboard_requested_model_contract.v1',
 'dashboard_export_path': 'research/dashboard_requested_model_contract_v1/dashboard_export.json',
 'dashboard_html_path': 'research/dashboard_requested_model_contract_v1/dashboard.html',
 'scheduled_dashboard_export_path': 'research/dashboard_requested_model_contract_v1/scheduled_dashboard_export.json',
 'scheduled_dashboard_html_path': 'research/dashboard_requested_model_contract_v1/scheduled_dashboard.html',
 'requested_model_contract': {'schema_version': 'model.requested_contract_health.v1',
  'requested_model_count': 32,
  'registered_model_count': 32,
  'ledger_model_count': 32,
  'completed_requested_model_count': 32,
  'missing_from_registry': [],
  'extra_in_registry': [],
  'missing_from_ledger': [],
  'incomplete_requested_models': [],
  'all_requested_models_completed': True,
  'adapter_statuses': ['paper_strict_adapter']},
 'schedule_payload': {'matrix_card': '/Users/rziiiii/dev/workspace/research/quantben

## 中文结论

- 做了什么：把 requested QuantBench 模型契约健康状态接入 dashboard export、view model、HTML 页面和 schedule refresh 通知路径。
- 核心指标：requested model count=32，registered model count=32，completed requested model count=32，missing_from_registry=[]，missing_from_ledger=[]，incomplete_requested_models=[]。
- regime 差异：本 notebook 不重新解释 regime 绩效，只复用 matrix card 已公开的 `regime_metrics`、`window_metrics`、`window_regime_metrics` 和 `model_review_briefing`；页面仍展示原有 regime/window 对比。
- 是否晋升：不晋升生产策略。本实验只补 dashboard/schedule 健康可见性。
- 风险：该健康项证明 32 个 requested base models 在当前 ledger 中完整，不证明短窗口收益可实盘外推；legacy card 没有 ledger 时不会误报 0/32。
- 下一步：如果要进入 production health，可以把该字段纳入生产 manifest 的 required dashboard checks，但仍保持 dashboard 只读 manifest/export。